<a href="https://colab.research.google.com/github/fabriciosantana/mcdia/blob/main/m4_exercicio_agente_compras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercício do Módulo IV
Complete o notebook para fazer com que o agente analisador de compras passe a considerar a média e o desvio padrão dos preços de referência para julgar se uma dada compra possui indicio de sobrepreço.

O agente só deve considerar sobrepreço se o preço estimado for maior que 10% da média dos preços de referência + 1/2 desvio padrão.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
csv_file = '/content/drive/MyDrive/Aulas/Aulas IDP/2026/Auditoria de Dados e Accountability/modulo IV - Grafos e IA/compras/itens_compras.csv'



In [ ]:
import pandas as pd

df_compras = pd.read_csv(csv_file, sep=";")
display(df_compras.head())

In [ ]:
compras_metadata = df_compras.to_json(orient='records', lines=True, force_ascii=False).splitlines()

print(f"Total de {len(compras_metadata)} linhas JSON geradas.")
print("Primeiras 5 linhas JSON:")
for i, line in enumerate(compras_metadata[:5]):
    print(f"Linha {i+1}: {line}")

In [ ]:
compras_indice = df_compras[["descricao",
                             "descricao_detalhada",
                             "unidade_fornecimento",
                             "codigo_item_catalogo"]].to_json(orient='records', lines=True, force_ascii=False).splitlines()

print(f"Total de {len(compras_indice)} linhas JSON geradas.")
print("Primeiras 5 linhas JSON:")
for i, line in enumerate(compras_indice[:5]):
    print(f"Linha {i+1}: {line}")

## Criando uma Base de Conhecimento Vetorial

Para criar uma base de conhecimento vetorial, seguiremos estes passos:
1.  **Instalar bibliotecas necessárias**: `chromadb` e `openai`.
2.  **Gerar embeddings**: Converter esses pedaços em representações vetoriais numéricas usando o modelo de embedding da OpenAI.
3.  **Criar o armazenamento vetorial**: Armazenar esses embeddings em uma instância ChromaDB, que permite buscas eficientes por similaridade.

In [ ]:
# Install necessary libraries for vector store and embeddings
!pip install -q chromadb openai

In [ ]:
!pip install -q langchain
!pip install -q langchain-openai
!pip install -q langchain-community

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma


In [ ]:
env_path = '/content/drive/MyDrive/Aulas/Aulas IDP/2026/Auditoria de Dados e Accountability/modulo IV - Grafos e IA/.env'

import json
from dotenv import load_dotenv
load_dotenv(env_path)

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document # Import Document class

# Initialize OpenAI Embeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Convert each JSON string in compras_json to a Document object
# Assuming each JSON string is a complete document, metadata can be added if available from the JSON.
documents_from_json = [Document(page_content=line, metadata=json.loads(meta)) for line, meta in zip(compras_indice, compras_metadata)]

# Create the vector store using ChromaDB from the Document objects
db = Chroma.from_documents(documents_from_json, embeddings, persist_directory='vectordb')

print("Vector database created successfully from compras_json!")

A base de dados vetorial `db` está agora pronta. Você pode usá-la para buscas por similaridade ou para recuperar informações relevantes.

## Compras Novas
As compras abaixo terão seus preços avaliados

In [ ]:
csv_file_path = '/content/drive/MyDrive/Aulas/Aulas IDP/2026/Auditoria de Dados e Accountability/modulo IV - Grafos e IA/compras/compras_novas.csv'
df_novas_compras = pd.read_csv(csv_file_path, sep=";")
df_novas_compras.head()


## Agente verificador de compras

### Estado do agente

In [ ]:
from typing import TypedDict, List, Optional, Annotated
from langchain_core.messages import BaseMessage, FunctionMessage
from langgraph.graph import add_messages
from pydantic import BaseModel

# Definir o modelo Pydantic para os detalhes do item
class ItemDetalhes(BaseModel):
    CHAVE_COMPRA_PNCP: str
    OBJETO: str
    numero_item: int
    descricao: str
    descricao_detalhada: str
    unidade_fornecimento: str
    valor_estimado: float
    quantidade_solicitada: int
    # Adicionando campos de metadados para melhor rastreamento
    NUMERO_UASG: Optional[str] = None
    NUMERO_COMPRA: Optional[str] = None
    ANO_COMPRA: Optional[int] = None

class ItemNaoSimilar(BaseModel):
    CHAVE_COMPRA_PNCP: str
    OBJETO: str
    numero_item: int
    descricao: str
    descricao_detalhada: str
    unidade_fornecimento: str
    valor_estimado: float
    quantidade_solicitada: int
    # Adicionando campos de metadados para melhor rastreamento
    NUMERO_UASG: Optional[str] = None
    NUMERO_COMPRA: Optional[str] = None
    ANO_COMPRA: Optional[int] = None
    JUSTIFICATIVA_DE_NAO_SIMILARIDADE: Optional[str]

# Definir o estado do agente
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


### Definição do agente

In [ ]:
from langchain.tools import tool
import pandas as pd
from typing import Optional

@tool
def get_pncp_key_by_index(index: int) -> Optional[ItemDetalhes]:
    """Retorna um objeto `ItemDetalhes` de um registro no arquivo 'compras_novas.csv' pelo índice.

    Args:
        index (int): O índice baseado em zero do registro no DataFrame para o qual o objeto `ItemDetalhes` deve ser retornado.

    Returns:
        Optional[ItemDetalhes]: O objeto `ItemDetalhes` correspondente ao índice fornecido, ou `None` se o índice for inválido,
             o arquivo não for encontrado, uma coluna necessária não existir ou ocorrer outro erro.
    """
    csv_file_path = '/content/drive/MyDrive/Aulas/Aulas IDP/2026/Auditoria de Dados e Accountability/modulo IV - Grafos e IA/compras/compras_novas.csv'
    try:
        df_novas_compras = pd.read_csv(csv_file_path, sep=";")
        if 0 <= index < len(df_novas_compras):
            row = df_novas_compras.loc[index]
            item_detalhes = ItemDetalhes(
                CHAVE_COMPRA_PNCP=row['CHAVE_COMPRA_PNCP'],
                OBJETO=row['OBJETO'],
                numero_item=int(row['numero_item']),
                descricao=row['descricao'],
                descricao_detalhada=row['descricao_detalhada'],
                unidade_fornecimento=row['unidade_fornecimento'],
                valor_estimado=float(row['valor_estimado']),
                quantidade_solicitada=int(row['quantidade_solicitada']),
                NUMERO_UASG= str(row['NUMERO_UASG']),
                NUMERO_COMPRA=str(row['NUMERO_COMPRA']),
                ANO_COMPRA=int(row['ANO_COMPRA'])
            )
            return item_detalhes
        else:
            print(f"Índice {index} fora dos limites do DataFrame.")
            return None
    except FileNotFoundError:
        print(f"Arquivo não encontrado: {csv_file_path}")
        return None
    except KeyError as e:
        print(f"Coluna não encontrada no arquivo CSV ou nome incorreto: {e}")
        return None
    except Exception as e:
        print(f"Ocorreu um erro ao carregar ou processar o arquivo: {e}")
        return None


## **Ponto de Mudança**
Substitua a Tool abaixo por uma tool que retorne a media e o desvio padrão da lista de preços.

Não se esqueça de ajustar a Docstring de acordo

In [ ]:
# Substituir
@tool
def calculate_median_price(prices: list[float]) -> float:
    """Calcula a mediana de uma lista de preços.

    Args:
        prices (list[float]): Uma lista de valores numéricos representando os preços.

    Returns:
        float: O valor da mediana dos preços fornecidos. Retorna None se a lista for vazia.
    """
    if not prices:
        return None
    import statistics
    return statistics.median(prices)


## **Nova Tool**
Adicione na célula abaixo, uma nova tool que verifica sobrepreço conforme as regras estabelecidas. Deve receber o preço do alvo, a média e o desvio padrão dos preços de referência e retornar "Possível sobrepreço" ou "Sobrepreço não identificado"

In [ ]:
@tool
def check_overpricing(
...
                       ) -> str:
    """Verifica ...
    """


### Realizando Geração Aumentada por Recuperação (RAG)

Agora vamos integrar a base de conhecimento vetorial com um modelo de linguagem para realizar o RAG. Isso permitirá que o modelo de linguagem use o contexto recuperado dos seus documentos para gerar respostas mais precisas e informadas à sua pergunta.

In [ ]:
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from typing import List # Ensure List is imported
from operator import itemgetter

# Initialize the LLM (Large Language Model)
llm = ChatOpenAI(model="gpt-5.4", temperature=0.1)

# Create a retriever from your ChromaDB instance, returning 15 items
retriever = db.as_retriever(search_kwargs={'k': 15})

# 1. Definir o modelo Pydantic para a lista de saída estruturada
class ListaDeItens(BaseModel):
    """Representa uma lista de itens de compra analisados encontrados na base de conhecimento."""
    itens_similares: List[ItemDetalhes]
    itens_nao_similares: List[ItemNaoSimilar]

# 2. Configurar o LLM para retornar saída estruturada
structured_llm = llm.with_structured_output(ListaDeItens)

# Helper function to format retrieved documents into a single string, including metadata
def format_docs_with_metadata(docs):
    formatted_docs = []
    for doc in docs:
        content = doc.page_content
        metadata = doc.metadata
        formatted_docs.append(f"--- Documento de Contexto ---\nConteúdo: {content}\nMetadados: {metadata}\n--------------------------")
    return "\n\n".join(formatted_docs)

# 3. Criar um prompt específico para RAG com saída estruturada
rag_prompt_structured = ChatPromptTemplate.from_messages([
    ("system", """Você é um assistente especialista em compras públicas. Sua tarefa é encontrar itens de compra semanticamente similares ao 'item_alvo' fornecido,
                  EXCLUSIVAMENTE dentro do contexto de compras públicas disponibilizado. Para cada item similar encontrado no contexto, extraia todas as informações necessárias para preencher o modelo ItemDetalhes.
                  Os metadados dos documentos de contexto estão disponíveis na seção 'Metadados:' de cada 'Documento de Contexto'.
                  Certifique-se de extrair `CHAVE_COMPRA_PNCP`, `NUMERO_UASG`, `NUMERO_COMPRA` e `ANO_COMPRA` dos metadados dos documentos de contexto, se disponíveis.
                  Voce deve retornar 2 listas ESTRITAMENTE seguindo o esquema `ListaDeItens`:
                    A: lista de itens similares - dos itens presentes no contexto, relacione aqui aqueles itens que são
                       similares ao item alvo.
                    B: Lista de itens não similares - dos itens presentes no contexto, relacione aqui aqueles que não sao similares para fins de cmparação de preço.    DOS ESTRITAMENTE seguindo o esquema `ListaDeItens`.
                  Não inclua nenhum item que não esteja explicitamente no contexto.
                  Se nenhum item similar for encontrado, retorne uma lista vazia.
                  Contexto de compras (com conteúdo e metadados):\n{context}
               """),
    ("user", "Com base no contexto, encontre todos os itens de compra similares ao seguinte item alvo: {input}"),
])

# 4. Criar uma cadeia de documentos estruturada usando LCEL
chain_structured = (
     rag_prompt_structured # Pass formatted context and input to the prompt
    | structured_llm # Pass the prompted input to the structured LLM
)

# A `retrieval_chain_structured` será removida e a lógica de recuperação será feita explicitamente na ferramenta.

@tool
def find_similar_items(item_alvo: dict) -> ListaDeItens:
    """Encontra itens de compra similares a um item alvo especificado, utilizando uma base de conhecimento vetorial.

    Args:
        item_alvo (dict): O item alvo para o qual buscar itens similares, contendo as informações necessárias:
                                  CHAVE_COMPRA_PNCP, OBJETO, descricao, descricao_detalhada, unidade_fornecimento, valor_estimado e quantidade_solicitada.

    Returns:
        ListaDeItensSimilares: Um objeto contendo uma lista de objetos ItemDetalhes que são considerados similares ao item alvo,
                               extraídos da base de conhecimento. Retorna uma lista vazia se nenhum item similar for encontrado
                               ou em caso de erro.
    """
    # Construir a string de consulta para a cadeia RAG com base nos atributos do item_alvo
    query_string = \
        f"Item alvo para busca de similares: Objeto: {item_alvo["OBJETO"]}, " +\
        f"Descrição: {item_alvo["descricao"]}, "+\
        f"Descrição Detalhada: {item_alvo["descricao_detalhada"]}, "+\
        f"Unidade de Fornecimento: {item_alvo["unidade_fornecimento"]}. " +\
        f"Valor Estimado: {item_alvo["valor_estimado"]}, Quantidade Solicitada: {item_alvo["quantidade_solicitada"]}."


    try:
        #print("Query para o retriever: <"+query_string+">")
        # 1. Recuperar documentos explicitamente
        retrieved_docs = retriever.invoke(query_string)
        #print(f"Documentos recuperados: {len(retrieved_docs)}")

        # 2. Formatar os documentos com metadados
        formatted_docs = format_docs_with_metadata(retrieved_docs)
        #print("Documentos formatados:")
        #print(formatted_docs)

        # 3. Invocar a cadeia de documentos estruturada com os documentos recuperados e a query
        response = chain_structured.invoke({"context": formatted_docs, "input": query_string})
        #print("Response do structured_llm: <"+str(response)+">")

        # A 'response' já é o objeto ListaDeItensSimilares diretamente do structured_llm
        return response
    except Exception as e:
        print(f"Erro ao buscar itens similares: {e}")
        return ListaDeItens(itens_similares=[])


## Construindo o Agente de Análise de Preços

### **Mudança**: Ajuste a lista de tools e o prompt para a nova especificação

In [ ]:
from typing import TypedDict, Annotated, List, Optional
from langchain_core.messages import BaseMessage, FunctionMessage, HumanMessage, AIMessage
from langchain_core.runnables import RunnablePassthrough
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode # Remover ToolExecutor da importação

# As classes ItemDetalhes, AgentState e as ferramentas (get_pncp_key_by_index, calculate_median_price, find_similar_items)
# são definidas em células anteriores e devem estar no escopo.
# O 'llm' e o 'retriever' também devem estar definidos em células anteriores.

# 1. Agrupar as ferramentas
tools = [
    ...
]

# 2. Criar um LLM com capacidade de chamada de ferramentas
agent_llm = llm.bind_tools(tools)

# 3. Definir o nó do agente que invoca o LLM
def agent_runner(state: AgentState):
    system_prompt = """Você é um agente especializado em auditoria de dados de compras públicas. Seu objetivo é ajudar o usuário a analisar o valor de um item alvo em comparação com preços medianos de itens similares.
                        Siga estas etapas:
                        1.  **Obter o Item Alvo**: O usuário fornecerá um índice para o item alvo. Use a ferramenta `get_pncp_key_by_index` para recuperar os detalhes completos desse item.
                        2.  **Encontrar Itens de Referência**: Com base nos detalhes do item alvo, use a ferramenta `find_similar_items` para encontrar uma lista de itens de compra similares na base de conhecimento.
                        3.  ...                       """
    prompt_template = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("placeholder", "{messages}"),
    ])

    chain = prompt_template | agent_llm

    messages = state['messages']
    # O agente invoca o LLM com as mensagens atuais. ToolNode cuidará dos resultados das ferramentas.
    response = chain.invoke({"messages": messages})
    #print(response)
    return {"messages": [response]}

# 5. Definir a lógica condicional para decidir o próximo passo
def should_continue(state: AgentState) -> str:
    messages = state['messages']
    last_message = messages[-1]
    # Se a última mensagem do agente inclui chamadas de ferramentas, vá para 'call_tool'.
    # Caso contrário, o agente chegou a uma conclusão e deve terminar.
    if last_message.tool_calls:
        return "call_tool"
    else:
        return "end_agent"

# 6. Construir o StateGraph
workflow = StateGraph(AgentState)

workflow.add_node("agent_node", agent_runner)
# Usar ToolNode padrão para executar as ferramentas
workflow.add_node("call_tool", ToolNode(tools=tools))

workflow.set_entry_point("agent_node")

workflow.add_conditional_edges(
    "agent_node",
    should_continue,
    {
        "call_tool": "call_tool",
        "end_agent": END
    },
)
# Após a execução da ferramenta, o fluxo retorna ao agent_node para o LLM processar o resultado da ferramenta
workflow.add_edge('call_tool', 'agent_node')

# 7. Compilar o gráfico em um agente executável
agent_executor = workflow.compile()

### Visualizando o Grafo

In [ ]:
from IPython.display import Image
display(Image(agent_executor.get_graph().draw_mermaid_png()))

### Executando o Agente

Agora você pode interagir com o agente fornecendo um índice para o item alvo. O agente usará suas ferramentas para realizar a análise e retornar o resultado.

In [ ]:
def run_agent_analysis(item_index: int):
    # Initialize the state for the agent with the user's request
    # The system message is already part of the agent_prompt defined above.
    # The agent_runner expects a list of messages for its input.
    # So, we start with a HumanMessage requesting the item at the given index.

    initial_input = {
        "messages": [HumanMessage(content=f"Analise o item com índice {item_index}.")]
    }

    # The AgentState needs to be correctly initialized for the graph.
    # The initial_input will be merged into the AgentState at the start.

    print(f"Iniciando análise para o item com índice: {item_index}")

    # Stream events from the agent to see its steps
    for s in agent_executor.stream(initial_input):
        if "agent_node" in s:
            for message in s["agent_node"]["messages"]:
                print(f"Agent: {message.content}")
                if message.tool_calls:
                    for tc in message.tool_calls:
                        # Acessar 'name' e 'args' como chaves de dicionário, pois 'tc' parece ser um dicionário
                        print(f"Agent calls tool: {tc['name']} with args {tc['args']}")
        elif "call_tool" in s:
            for message in s["call_tool"]["messages"]:
                print(f"Tool Output: {message.content}")
        elif END in s:
            final_state = s[END]
            final_message = final_state['messages'][-1]
            print("\n--- Análise Finalizada ---")
            print(final_message.content)


# Exemplo de uso do agente:
# run_agent_analysis(item_index=0) # Altere o índice conforme necessário para testar

In [ ]:
run_agent_analysis(item_index=57)